In [13]:
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

In [14]:
assert load_dotenv(find_dotenv(), override=True)
client = OpenAI()

- https://developers.openai.com/cookbook/examples/responses_api/reasoning_items
    - CoT 继承；

### why response api

- 对于 gpt-5 / 推理模型，Responses 还有“更容易把上一轮的推理状态延续到下一轮”的机制（内部保留 reasoning items，不暴露给你），这也是官方更建议迁移的原因之一。
    - 因为 Responses 是 stateful 的，这些 reasoning tokens 会“跟着链路走”。更关键的是：如果你用 previous_response_id，模型会自动拥有之前产生的所有 reasoning items，你不需要手动塞回去。
- Responses：items 是多态列表，天然表达“模型说了什么/做了什么/调用了什么工具/工具输出是什么”，并且能用 previous_response_id 让链路延续（包括 reasoning items）。

In [26]:
def get_fx_rate(base: str, quote: str) -> float:
    return 1.10  # demo

tools = [{
    "type": "function",
    "name": "get_fx_rate",
    "description": "Get FX rate for base->quote",
    "parameters": {
        "type": "object",
        "properties": {
            "base": {"type": "string"},
            "quote": {"type": "string"},
        },
        "required": ["base", "quote"],
        "additionalProperties": False,
    },
    "strict": True,
}]

input_list = [{"role": "user", "content": "把 1234 EUR 换成 USD。不要猜汇率，先调用 get_fx_rate。"}]

r1 = client.responses.create(model="gpt-5-mini", tools=tools, input=input_list)

In [27]:
from rich.pretty import pprint
pprint(r1)

Response(
│   id='resp_05ce74ac71e4deae00699dbfdbc968819091cbfa85f6c808b3',
│   created_at=1771945947.0,
│   error=None,
│   incomplete_details=None,
│   instructions=None,
│   metadata={},
│   model='gpt-5-mini-2025-08-07',
│   object='response',
│   output=[
│   │   ResponseReasoningItem(
│   │   │   id='rs_05ce74ac71e4deae00699dbfdc5a148190a68c9c6208c72686',
│   │   │   summary=[],
│   │   │   type='reasoning',
│   │   │   content=None,
│   │   │   encrypted_content=None,
│   │   │   status=None
│   │   ),
│   │   ResponseFunctionToolCall(
│   │   │   arguments='{"base":"EUR","quote":"USD"}',
│   │   │   call_id='call_rZrmjuM4gfeA0LCaPelSazJU',
│   │   │   name='get_fx_rate',
│   │   │   type='function_call',
│   │   │   id='fc_05ce74ac71e4deae00699dbfde76b0819090992b7958476403',
│   │   │   status='completed'
│   │   )
│   ],
│   parallel_tool_calls=True,
│   temperature=1.0,
│   tool_choice='auto',
│   tools=[
│   │   FunctionTool(
│   │   │   name='get_fx_rate',
│   │   │   parameters={
│   │   │   │   'type': 'object',
│   │   │   │   'properties': {'base': {'type': 'string'}, 'quote': {'type': 'string'}},
│   │   │   │   'required': ['base', 'quote'],
│   │   │   │   'additionalProperties': False
│   │   │   },
│   │   │   strict=True,
│   │   │   type='function',
│   │   │   description='Get FX rate for base->quote'
│   │   )
│   ],
│   top_p=1.0,
│   background=False,
│   conversation=None,
│   max_output_tokens=None,
│   max_tool_calls=None,
│   previous_response_id=None,
│   prompt=None,
│   prompt_cache_key=None,
│   prompt_cache_retention=None,
│   reasoning=Reasoning(effort='medium', generate_summary=None, summary=None),
│   safety_identifier=None,
│   service_tier='default',
│   status='completed',
│   text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity='medium'),
│   top_logprobs=0,
│   truncation='disabled',
│   usage=ResponseUsage(
│   │   input_tokens=70,
│   │   input_tokens_details=InputTokensDetails(cached_tokens=0),
│   │   output_tokens=174,
│   │   output_tokens_details=OutputTokensDetails(reasoning_tokens=128),
│   │   total_tokens=244
│   ),
│   user=None,
│   billing={'payer': 'openai'},
│   completed_at=1771945950,
│   frequency_penalty=0.0,
│   presence_penalty=0.0,
│   store=True
)

In [28]:
r1.output

[ResponseReasoningItem(id='rs_05ce74ac71e4deae00699dbfdc5a148190a68c9c6208c72686', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"base":"EUR","quote":"USD"}', call_id='call_rZrmjuM4gfeA0LCaPelSazJU', name='get_fx_rate', type='function_call', id='fc_05ce74ac71e4deae00699dbfde76b0819090992b7958476403', status='completed')]

In [29]:
import json
input_list += r1.output

for item in r1.output:
    if item.type == "function_call" and item.name == "get_fx_rate":
        args = json.loads(item.arguments)
        rate = get_fx_rate(**args)
        input_list.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps({"rate": rate}),
        })

r2 = client.responses.create(model="gpt-5", tools=tools, input=input_list)
print(r2.output_text)

按实时汇率 EUR→USD = 1.1 计算：
1234 EUR × 1.1 = 1,357.40 USD

需要我用不同小数位或四舍五入规则吗？


In [31]:
pprint(r2)

Response(
│   id='resp_05ce74ac71e4deae00699dbfe620288190a3f83ec167ad5b18',
│   created_at=1771945958.0,
│   error=None,
│   incomplete_details=None,
│   instructions=None,
│   metadata={},
│   model='gpt-5-2025-08-07',
│   object='response',
│   output=[
│   │   ResponseReasoningItem(
│   │   │   id='rs_05ce74ac71e4deae00699dbfe6cde08190b45a3dfd20548fa3',
│   │   │   summary=[],
│   │   │   type='reasoning',
│   │   │   content=None,
│   │   │   encrypted_content=None,
│   │   │   status=None
│   │   ),
│   │   ResponseOutputMessage(
│   │   │   id='msg_05ce74ac71e4deae00699dbfeacf908190966843bd38b51f9e',
│   │   │   content=[
│   │   │   │   ResponseOutputText(
│   │   │   │   │   annotations=[],
│   │   │   │   │   text='按实时汇率 EUR→USD = 1.1 计算：\n1234 EUR × 1.1 = 1,357.40 USD\n\n需要我用不同小数位或四舍五入规则吗？',
│   │   │   │   │   type='output_text',
│   │   │   │   │   logprobs=[]
│   │   │   │   )
│   │   │   ],
│   │   │   role='assistant',
│   │   │   status='completed',
│   │   │   type='message'
│   │   )
│   ],
│   parallel_tool_calls=True,
│   temperature=1.0,
│   tool_choice='auto',
│   tools=[
│   │   FunctionTool(
│   │   │   name='get_fx_rate',
│   │   │   parameters={
│   │   │   │   'type': 'object',
│   │   │   │   'properties': {'base': {'type': 'string'}, 'quote': {'type': 'string'}},
│   │   │   │   'required': ['base', 'quote'],
│   │   │   │   'additionalProperties': False
│   │   │   },
│   │   │   strict=True,
│   │   │   type='function',
│   │   │   description='Get FX rate for base->quote'
│   │   )
│   ],
│   top_p=1.0,
│   background=False,
│   conversation=None,
│   max_output_tokens=None,
│   max_tool_calls=None,
│   previous_response_id=None,
│   prompt=None,
│   prompt_cache_key=None,
│   prompt_cache_retention=None,
│   reasoning=Reasoning(effort='medium', generate_summary=None, summary=None),
│   safety_identifier=None,
│   service_tier='default',
│   status='completed',
│   text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity='medium'),
│   top_logprobs=0,
│   truncation='disabled',
│   usage=ResponseUsage(
│   │   input_tokens=113,
│   │   input_tokens_details=InputTokensDetails(cached_tokens=0),
│   │   output_tokens=250,
│   │   output_tokens_details=OutputTokensDetails(reasoning_tokens=192),
│   │   total_tokens=363
│   ),
│   user=None,
│   billing={'payer': 'openai'},
│   completed_at=1771945963,
│   frequency_penalty=0.0,
│   presence_penalty=0.0,
│   store=True
)

### Server-side Code Interpreter

- Zero client setup, secure sandboxed execution, transparent audit trail.

In [5]:
response = client.responses.create(
    model="gpt-4o-mini",
    tools=[{"type": "code_interpreter", 
            "container": {"type": "auto"}
           }],
    input='Generate 5 random numbers and calculate standard deviation.'
)

In [7]:
print(response.output_text)

The 5 random numbers generated are:

- 0.4632
- 0.1928
- 0.0788
- 0.8632
- 0.3489

The standard deviation of these numbers is approximately **0.2708**.


In [10]:
from rich.pretty import pprint
pprint(response.output)

[
│   ResponseCodeInterpreterToolCall(
│   │   id='ci_09b204efc0c686420069858c44457481968229d08845df1a25',
│   │   code='import numpy as np\r\n\r\n# Generate 5 random numbers\r\nrandom_numbers = np.random.random(5)\r\n\r\n# Calculate standard deviation\r\nstd_deviation = np.std(random_numbers)\r\n\r\nrandom_numbers, std_deviation',
│   │   container_id='cntr_69858c4335bc8193a4fef764449b582b01083fccb29009f5',
│   │   outputs=None,
│   │   status='completed',
│   │   type='code_interpreter_call'
│   ),
│   ResponseOutputMessage(
│   │   id='msg_09b204efc0c686420069858c4d5b848196a95a5ee16b4de7b3',
│   │   content=[
│   │   │   ResponseOutputText(
│   │   │   │   annotations=[],
│   │   │   │   text='The 5 random numbers generated are:\n\n- 0.4632\n- 0.1928\n- 0.0788\n- 0.8632\n- 0.3489\n\nThe standard deviation of these numbers is approximately **0.2708**.',
│   │   │   │   type='output_text',
│   │   │   │   logprobs=[]
│   │   │   )
│   │   ],
│   │   role='assistant',
│   │   status='completed',
│   │   type='message'
│   )
]

In [12]:
print(response.output[0].code)

import numpy as np

# Generate 5 random numbers
random_numbers = np.random.random(5)

# Calculate standard deviation
std_deviation = np.std(random_numbers)

random_numbers, std_deviation
